<a href="https://colab.research.google.com/github/jkh02/-XAI-LLM-/blob/main/XAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# 2. 내 드라이브 내 프로젝트 전용 폴더 생성 및 이동
# 'MyProject' 대신 본인이 원하는 폴더명을 적으세요.
project_path = "/content/drive/MyDrive/MyProject"

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f" 새 폴더를 생성했습니다: {project_path}")

os.chdir(project_path)
print(f" 현재 작업 위치: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📍 현재 작업 위치: /content/drive/MyDrive/MyProject


In [ ]:
# 1. 1.24.4 대신 <2.0.0으로 설정하여 빌드 에러를 방지하고 spacy 호환성을 맞춥니다.
!pip install "numpy<2.0.0" spacy

# 아까 덜 깔린 부품들만 다시 깔아줍니다. (넘파이는 이미 해결됐으니 패스!)
!pip install -q transformers peft accelerate bitsandbytes captum langchain langchain-community pypdf faiss-cpu sentence-transformers datasets
print(" 이제  준비되었습니다!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
✅ 이제  준비되었습니다!


In [ ]:
import os
import json
import torch
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from captum.attr import LayerIntegratedGradients
from datasets import Dataset
from datetime import datetime

# 경로 기본 세팅
project_path = "/content/drive/MyDrive/MyProject"
save_path = os.path.join(project_path, "saved_weights")

# ==========================================
#  1단계: JSONL 데이터 로드 및 지식 창고 생성
# ==========================================
print(" 임베딩 모델을 준비합니다...")
embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")

# 압축 해제한 파일이 있는 경로
jsonl_file_path = os.path.join(project_path, "data1", "chunks.jsonl")

docs = []
print(" JSONL 파일에서 가공된 데이터를 읽어옵니다...")

try:
    with open(jsonl_file_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            doc = Document(
                page_content=item["chunk_text"],
                metadata={
                    "title": item.get("title", ""),
                    "doc_id": item.get("doc_id", ""),
                    "source": item.get("source_url", "")
                }
            )
            docs.append(doc)
    print(f" 총 {len(docs)}개의 가공된 문서 청크를 성공적으로 불러왔습니다!")

except Exception as e:
    print(f" 데이터 로드 실패: {e}")

if docs:
    print(" 지식 창고(FAISS)를 구축 중입니다...")
    vectorstore = FAISS.from_documents(documents=docs, embedding=embeddings)

    faiss_save_path = os.path.join(project_path, "saved_faiss_index_json")
    vectorstore.save_local(faiss_save_path)
    print(" 구축된 지식 창고를 드라이브에 저장했습니다.")

# ==========================================
#  2단계: AI 모델 및 XAI(Captum) 세팅
# ==========================================
print("⏳ 2단계: AI 모델 및 가중치 세팅 중...")
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

if os.path.exists(os.path.join(save_path, "adapter_config.json")):
    print(" 드라이브에 저장된 기존 학습 가중치를 불러옵니다.")
    from peft import PeftModel
    model = PeftModel.from_pretrained(base_model, save_path, is_trainable=True)
else:
    print(" 저장된 가중치가 없습니다. 새로운 LoRA 어댑터를 생성합니다.")
    config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
    model = get_peft_model(base_model, config)

target_layer = model.base_model.model.model.embed_tokens
lig = LayerIntegratedGradients(model, target_layer)

# ==========================================
#  2.5단계: 데이터 변환 및 모델 학습 (Fine-Tuning)
# ==========================================
print("⏳ 학습을 위한 데이터 변환을 시작합니다...")

#  버그 수정: splits 대신 docs 사용!
train_texts = [doc.page_content for doc in docs]

dataset = Dataset.from_dict({"text": train_texts})

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 날짜 자동 생성 및 체크포인트 폴더 설정
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_dir = os.path.join(project_path, f"checkpoints_{timestamp}")
print(f"이번 학습 과정은 '{checkpoint_dir}'에 분리되어 저장됩니다.")

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=10,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("모델 파인튜닝을 시작합니다")
trainer.train()

# ==========================================
#  2.6단계: 최종 학습된 모델 드라이브에 저장
# ==========================================
if not os.path.exists(save_path):
    os.makedirs(save_path)
trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("모델 드라이브 저장 완료")

⏳ 임베딩 모델을 준비합니다...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ JSONL 파일에서 가공된 데이터를 읽어옵니다...
✅ 총 100개의 가공된 문서 청크를 성공적으로 불러왔습니다!
⏳ 지식 창고(FAISS)를 구축 중입니다...
💾 구축된 지식 창고를 드라이브에 저장했습니다.
⏳ 2단계: AI 모델 및 가중치 세팅 중...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

🔄 드라이브에 저장된 기존 학습 가중치를 불러옵니다.


RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: copying a param with shape torch.Size([16, 2048]) from checkpoint, the shape in current model is torch.Size([16, 1536]).
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: copying a param with shape torch.Size([2048, 16]) from checkpoint, the shape in current model is torch.Size([1536, 16]).
	size mismatch for base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: copying a param with shape torch.Size([16, 2048]) from checkpoint, the shape in current model is torch.Size([16, 1536]).
	size mismatch for base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight: copying a param with shape torch.Size([16, 2048]) from checkpoint, the shape in current model is torch.Size([16, 1536]).
	size mismatch for base_model.model.model.layers.1.self_attn.q_proj.lora_B.defaul

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ==========================================
#  1. 저장된 데이터와 모델 '불러오기'
# ==========================================
print("저장된 데이터와 모델을 불러오는 중입니다...")

project_path = "/content/drive/MyDrive/MyProject"
faiss_save_path = os.path.join(project_path, "saved_faiss_index_json")
save_path = os.path.join(project_path, "saved_weights")

# 1-1. FAISS 즉시 로드
embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")
vectorstore = FAISS.load_local(faiss_save_path, embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 1-2. 기본 모델에 저장된 '가중치' 덧씌우기
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

model = PeftModel.from_pretrained(base_model, save_path)
print(" 챗봇 준비 완료!\n")

# ==========================================
#  2. 대화형 질문 및 답변 생성
# ==========================================
print("\n==================================================")
print("복지 정책 챗봇이 준비되었습니다! (종료하려면 '종료' 또는 'q' 입력)")
print("==================================================")

# 질문 입력
while True:
    question = input("\n 질문을 입력하세요: ")

    if question.lower() in ['종료', 'q', 'quit', 'exit']:
        print("종료합니다.")
        break

    if not question.strip():
        continue

    # 문서 검색
    docs = retriever.invoke(question)

    context_list = []
    for doc in docs:
        title = doc.metadata.get("title", "제목 없음")
        content = doc.page_content
        context_list.append(f"[정책명: {title}]\n내용: {content}")

    context = "\n\n".join(context_list)

    # TinyLlama 전용 대화형 프롬프트 적용
    prompt = f"<|system|>\n당신은 대한민국의 친절한 복지 정책 안내 챗봇입니다. 주어진 문서의 내용만을 바탕으로 명확하고 자연스러운 한국어로 답변하세요.</s>\n<|user|>\n다음 문서를 참고하여 질문에 답해주세요.\n\n[문서]\n{context}\n\n질문: {question}</s>\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    print("답변을 찾고 있습니다...\n")


# 모델 추론 (파라미터 정상화)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

    # 프롬프트 길이만큼을 제외하고 새로 생성된 토큰만 정확히 추출
    input_length = inputs.input_ids.shape[1] # 질문의 길이
    generated_tokens = outputs[0][input_length:] # 전체 결과에서 질문 길이 이후부터만 가져옴

    # 생성한 답변만 텍스트로 변환
    final_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print("--------------------------------------------------")
    print(final_answer)
    print("--------------------------------------------------")

저장된 데이터와 모델을 불러오는 중입니다...


/tmp/ipykernel_16073/226050018.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ 챗봇 준비 완료!


복지 정책 챗봇이 준비되었습니다! (종료하려면 '종료' 또는 'q' 입력)

 질문을 입력하세요: 저소득층을 위한 복지정책 추천해줘
답변을 찾고 있습니다...

--------------------------------------------------
[질문]
- 양곡할인 정책이 무엇인지 설명해드리고, 양곡을 기초수급가구 및 차상위계층에 할인로 사용할 수 있는 방식을 보여줌으로써 저소득층의 생활안정을 도모합니까?
--------------------------------------------------


KeyboardInterrupt: Interrupted by user